# 🌍 World Bank ESMAP Multi-Tier Framework (MTF) Interactive Analytical Suite
**Open-Source Data Exploration, Crosstabs, Policy Simulations & Exports**

This turnkey notebook enables researchers, economists, and operational team members without a Stata license to independently explore MTF household survey data, calculate survey-weighted statistics, run policy simulations, and export publication-ready tables and charts.

### 🚀 How to Run:
- **Local (JupyterLab / VS Code):** Simply run the cells sequentially (Shift + Enter).
- **Google Colab:** Click `Open in Colab`, upload your `MTF_Master.dta` (or mount Google Drive), and run.
---

In [ ]:
# 1. Environment Setup & Library Imports
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Publication styling parameters (World Bank palette)
WB_NAVY = '#002244'
WB_TEAL = '#008080'
WB_ORANGE = '#f39c12'
WB_CORAL = '#d9534f'
WB_GRAY = '#7f8c8d'

plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['figure.dpi'] = 150

print('✓ Libraries loaded successfully.')

In [ ]:
# 2. Universal Data Ingestion (Local Disk, Media Mount, or Google Drive)
candidate_paths = [
    Path('Data/MTF_2024_HH_Guinea-Bissau_Master.dta'),
    Path('Data/wip_hh/MTF_2024_HH_Guinea-Bissau_Master.dta'),
    Path('./MTF_2024_HH_Guinea-Bissau_Master.dta'),
    Path('../Data/wip_hh/MTF_2024_HH_Guinea-Bissau_Master.dta'),
    Path('/media/bryan/Bryan/MTF/9_Data analysis_MTF countries/37_Guinea-Bissau/Data/wip_hh/MTF_2024_HH_Guinea-Bissau_Master.dta'),
    Path('/content/drive/MyDrive/MTF_2024_HH_Guinea-Bissau_Master.dta')
]

# Auto-detect any .dta dropped into Data/ or current folder
for search_dir in [Path('Data'), Path('Data/wip_hh'), Path('.')]:
    if search_dir.exists():
        for dta_f in search_dir.glob('*.dta'):
            if dta_f not in candidate_paths:
                if 'master' in dta_f.name.lower():
                    candidate_paths.insert(0, dta_f)
                else:
                    candidate_paths.append(dta_f)

data_file = None
# First search for master files
for p in candidate_paths:
    p_obj = Path(p)
    if p_obj.exists() and 'master' in p_obj.name.lower():
        data_file = str(p_obj)
        break

if not data_file:
    for p in candidate_paths:
        p_obj = Path(p)
        if p_obj.exists():
            data_file = str(p_obj)
            break

if data_file:
    print(f'[*] Ingesting dataset from: {data_file}')
    df = pd.read_stata(data_file, convert_categoricals=False)
    print(f'✓ Successfully loaded {len(df):,} households and {len(df.columns):,} variables.')
else:
    raw_sections = list(Path('Data').glob('section_*.dta'))
    if raw_sections:
        print('⚠️ Notice: Found raw section files in Data/, but MTF Master dataset (MTF_2024_HH_*_Master.dta) is needed for analysis.')
        print('   Please place the compiled master dataset in interactive_suite/Data/ or specify its path.')
    else:
        print('⚠️ Please place MTF Master .dta file in Data/ folder or specify data_file path.')

In [ ]:
# Helper: Survey-Weighted Mean & Proportions
def svy_mean(series, weights, mask=None):
    if mask is not None:
        s = series[mask]
        w = weights[mask]
    else:
        s = series
        w = weights
    valid = (~s.isna()) & (~w.isna()) & (w > 0)
    s = s[valid]
    w = w[valid]
    if len(s) == 0 or np.sum(w) == 0:
        return 0.0
    return float(np.sum(s * w) / np.sum(w))

## 📊 Module 1: Multi-Dimensional Crosstabs & Disaggregation
Analyze access rates and Tier distributions across demographic, geographic, and income partitions.

In [ ]:
# Multi-Tier Electricity & Cooking Disaggregation by Urban/Rural and Gender
w = df['hh_wt']

gender_col = 'hh_head_gender' if 'hh_head_gender' in df.columns else 'gender_head'

groups = {
    'National': np.ones(len(df), dtype=bool),
    'Urban': df['loc_urban_rural'] == 1,
    'Rural': df['loc_urban_rural'] == 2,
    'Male-Headed': df[gender_col] == 1 if gender_col in df.columns else np.zeros(len(df), dtype=bool),
    'Female-Headed': df[gender_col] == 2 if gender_col in df.columns else np.zeros(len(df), dtype=bool)
}

table_rows = []
for grp_name, mask in groups.items():
    elec_acc = svy_mean((df['elc_aggr_tier'] >= 1).astype(float), w, mask) * 100
    elec_t45 = svy_mean((df['elc_aggr_tier'] >= 4).astype(float), w, mask) * 100
    grid_acc = svy_mean((df['elec_tech_type'] == 1).astype(float), w, mask) * 100
    solar_acc = svy_mean(df['elec_tech_type'].isin([4, 5, 6]).astype(float), w, mask) * 100
    cook_mecs = svy_mean((df['cs_aggr_tier'] >= 4).astype(float), w, mask) * 100
    sample_n = int(mask.sum())
    
    table_rows.append({
        'Disaggregation': grp_name,
        'Sample (N)': sample_n,
        'Electricity Tier 1+ (%)': round(elec_acc, 1),
        'High Tier 4+ (%)': round(elec_t45, 1),
        'National Grid (%)': round(grid_acc, 1),
        'Off-Grid Solar (%)': round(solar_acc, 1),
        'Clean Cooking MECS (%)': round(cook_mecs, 1)
    })

summary_df = pd.DataFrame(table_rows)
display(summary_df)

# Plot Multi-Tier Disaggregation Comparison
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(summary_df))
width = 0.35
ax.bar(x - width/2, summary_df['Electricity Tier 1+ (%)'], width, label='Electricity Access (Tier 1+)', color=WB_TEAL)
ax.bar(x + width/2, summary_df['Clean Cooking MECS (%)'], width, label='Clean Cooking (Tier 4+)', color=WB_ORANGE)
ax.set_xticks(x)
ax.set_xticklabels(summary_df['Disaggregation'])
ax.set_ylabel('Access Rate (%)')
ax.set_title('Energy Access Disaggregation by Sector and Household Headship', fontweight='bold', color=WB_NAVY)
ax.legend(frameon=False)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 🎯 Module 2: Deficit & Vulnerability Targeting Filters
Isolate and profile critical policy target populations:

In [ ]:
# Filter 1: Under-the-Grid Deficit (Living under grid lines but unconnected)
under_grid_unconnected = (df['ea_grid_present'] == 1) & (df['grid_hh'] == 0)
pct_under_grid_deficit = svy_mean(under_grid_unconnected.astype(float), w) * 100
n_under_grid_deficit = under_grid_unconnected.sum()

# Filter 2: Biomass-Stacking Clean Stove Owners
clean_col = 'clean_stove' if 'clean_stove' in df.columns else 'clean_primary_cook'
clean_owners = df[clean_col] == 1
stackers = (df[clean_col] == 1) & (df['stacking_dilution'] == 1)
pct_stacking_among_clean = (svy_mean(stackers.astype(float), w) / svy_mean(clean_owners.astype(float), w)) * 100

print(f'📌 [Target Group 1] Under-the-Grid Unconnected Deficit:')
print(f'   - Sample Households: {n_under_grid_deficit:,} households')
print(f'   - National Share: {pct_under_grid_deficit:.1f}% of national population lives under grid lines without connection!')

print(f'📌 [Target Group 2] Clean Stove Stacking Dilution:')
print(f'   - {pct_stacking_among_clean:.1f}% of households owning clean cookstoves continue using traditional biomass!')

## ⚡ Module 3: Policy Scenario Simulator
Simulate the effect of electricity tariff adjustments and connection financing subsidies.

In [ ]:
# Scenario Simulator: Electricity Tariff Impact on Basic Bundle Affordability
# Baseline EAGB Tariff = 128 XOF/kWh. Basic bundle = 365 kWh/year (Tier 3 threshold: 5% of expenditure)
BASE_TARIFF = 128.0
KWH_BUNDLE = 365.0

scenarios = [-0.30, -0.15, 0.0, 0.15, 0.30, 0.50]  # Tariff shocks (-30% to +50%)
exp_col = 'exp_total_yr' if 'exp_total_yr' in df.columns else ('exp_annual_hh' if 'exp_annual_hh' in df.columns else 'exp_m_hh')
q_col = 'hh_exp_quintile' if 'hh_exp_quintile' in df.columns else 'quintile'

sim_results = []
if exp_col in df.columns:
    annual_exp = df[exp_col] if exp_col in ['exp_total_yr', 'exp_annual_hh'] else df[exp_col] * 12.0
    
    for shock in scenarios:
        t_rate = BASE_TARIFF * (1.0 + shock)
        bundle_cost = KWH_BUNDLE * t_rate
        # Affordable if cost is <= 5% of total household consumption expenditure
        affordable = (bundle_cost / annual_exp) <= 0.05
        pct_afford_nat = svy_mean(affordable.astype(float), w) * 100
        pct_afford_q1 = svy_mean(affordable.astype(float), w, df[q_col] == 1) * 100 if q_col in df.columns else 0.0
        
        sim_results.append({
            'Scenario Shock': f'{shock*100:+.0f}%',
            'Tariff (XOF/kWh)': round(t_rate, 1),
            'Annual Bundle Cost (XOF)': round(bundle_cost, 0),
            'National Affordability (%)': round(pct_afford_nat, 1),
            'Poorest Quintile (Q1) Affordability (%)': round(pct_afford_q1, 1)
        })
    
    sim_df = pd.DataFrame(sim_results)
    display(sim_df)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(sim_df['Tariff (XOF/kWh)'], sim_df['National Affordability (%)'], marker='o', color=WB_TEAL, linewidth=2, label='National Affordability')
    ax.plot(sim_df['Tariff (XOF/kWh)'], sim_df['Poorest Quintile (Q1) Affordability (%)'], marker='s', color=WB_CORAL, linewidth=2, linestyle='--', label='Poorest Quintile (Q1)')
    ax.axvline(BASE_TARIFF, color=WB_GRAY, linestyle=':', label='Current Tariff (128 XOF)')
    ax.set_xlabel('Electricity Tariff (XOF / kWh)')
    ax.set_ylabel('Households Able to Afford Basic Bundle (%)')
    ax.set_title('Policy Simulation: Electricity Tariff vs Basic Bundle Affordability', fontweight='bold', color=WB_NAVY)
    ax.legend(frameon=False)
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

## 💾 Module 4: One-Click Excel Export Suite
Export custom cross-tabulations and simulation results to formatted Excel files.

In [ ]:
# Export Analysis Results to Excel
output_excel = 'MTF_Interactive_Analysis_Output.xlsx'
with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='Disaggregations', index=False)
    if 'sim_df' in locals():
        sim_df.to_excel(writer, sheet_name='Tariff_Simulation', index=False)

print(f'✓ Export complete! Formatted analysis saved to: {output_excel}')

## 🩺 Module 5: 1-Click System Diagnostics & Support Log
If you encounter any error or unexpected behavior, run this cell to automatically generate a diagnostic report (`mtf_diagnostic_report.txt`).
You can simply attach this file to an email and send it to Bryan Koo (bkoo@worldbankgroup.org) for instant debugging.

In [ ]:
# Automated Diagnostic Log Generator
import platform
import datetime

diag_file = 'mtf_diagnostic_report.txt'
with open(diag_file, 'w', encoding='utf-8') as f:
    f.write('=================================================================\n')
    f.write('   World Bank ESMAP MTF Interactive Analyzer - Diagnostic Log    \n')
    f.write('=================================================================\n')
    f.write(f'Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
    f.write(f'Operating System: {platform.platform()} ({platform.system()})\n')
    f.write(f'Python Executable: {sys.executable}\n')
    f.write(f'Python Version: {sys.version.split()[0]}\n')
    f.write(f'Current Directory: {Path.cwd()}\n\n')
    
    f.write('--- [1] Environment Libraries ---\n')
    for pkg in ['pandas', 'numpy', 'matplotlib', 'openpyxl', 'jupyterlab']:
        try:
            mod = __import__(pkg)
            f.write(f'  ✓ {pkg}: {getattr(mod, "__version__", "installed")}\n')
        except ImportError:
            f.write(f'  ✗ {pkg}: NOT INSTALLED\n')
            
    f.write('\n--- [2] Data File Existence & Permissions ---\n')
    for cand in candidate_paths:
        cand_p = Path(cand)
        status = 'FOUND' if cand_p.exists() else 'NOT FOUND'
        sz_str = f'({cand_p.stat().st_size / 1024 / 1024:.2f} MB)' if cand_p.exists() else ''
        f.write(f'  [{status}] {cand_p} {sz_str}\n')
        
    f.write('\n--- [3] Dataset Integrity ---\n')
    try:
        if 'df' in locals() and df is not None:
            f.write(f'  Loaded DataFrame Shape: {df.shape[0]} households, {df.shape[1]} variables\n')
            w_found = next((c for c in ['hh_wt', 'weight', 'wt', 'hhweight'] if c in df.columns), None)
            if w_found:
                f.write(f'  Weight Variable: {w_found} (Total Weighted Households: {df[w_found].sum():,.0f})\n')
            else:
                f.write('  Weight Variable: None detected in master dataset\n')
            u_found = next((c for c in ['loc_urban_rural', 'urban_rural', 'urban', 'strata'] if c in df.columns), None)
            if u_found:
                u_count = (df[u_found] == 1).sum()
                f.write(f'  Urban Variable: {u_found} (Urban HHs: {u_count:,})\n')
            else:
                f.write('  Urban Variable: None detected in master dataset\n')
            t_elec = 'elc_aggr_tier' in df.columns
            t_cook = 'cs_aggr_tier' in df.columns or 'cook_aggr_tier' in df.columns
            f.write(f'  Electricity Tier Variable: {"FOUND (elc_aggr_tier)" if t_elec else "MISSING"}\n')
            f.write(f'  Clean Cooking Tier Variable: {"FOUND (cs_aggr_tier)" if t_cook else "MISSING"}\n')
        else:
            f.write('  DataFrame status: Not loaded or loading failed.\n')
    except Exception as e_diag:
        f.write(f'  Integrity check note: {e_diag}\n')
    f.write('=================================================================\n')

print('=' * 65)
print(f'✓ Diagnostic log successfully generated: {diag_file}')
print('  Please send this text file to Bryan Koo (bkoo@worldbankgroup.org)')
print('  if you need assistance or encounter an error!')
print('=' * 65)